# Día 3 · Unidad 3 — Leer errores y primer debugging (con y sin IA)

**Objetivos de aprendizaje**
- Leer un traceback de Python de arriba abajo con criterio, en vez de asumir "algo está roto en todo el bloque".
- Reconocer los tipos de error más frecuentes al escribir condicionales y bucles: `NameError`, `TypeError`, `KeyError`, `IndexError`, `ZeroDivisionError`, además de los ya vistos hoy (`IndentationError`, `SyntaxError`, `StopIteration`).
- Usar Claude Code como apoyo para interpretar un traceback y proponer una corrección — sin copiar la respuesta sin entenderla.

**Duración estimada:** 45-60 minutos.

**Requisitos previos:** los cuatro notebooks anteriores de este mismo día (condicionales, `while`, `for`, comprehensions).

## 1. Anatomía de un traceback

Cuando un programa Python falla, no imprime solo "ha habido un error": imprime un **traceback**, que es literalmente el camino que siguió el error hasta llegar a la superficie. Tiene una estructura fija, y aprender a leerla en el orden correcto es la habilidad de depuración más rentable de todo el curso.

Se lee así:

1. **De arriba abajo**, cada bloque `File "...", line N, in <función>` es una "parada" de la llamada: qué archivo, qué línea, dentro de qué función se estaba ejecutando en ese punto.
2. La **última** parada (la de más abajo) es el sitio exacto donde el error ocurrió — casi siempre el punto de partida para investigar.
3. La **última línea del todo** (`TipoDeError: mensaje`) te dice **qué** tipo de fallo es y, casi siempre, un mensaje concreto sobre qué esperaba Python y qué encontró.

Vamos a provocar un error real con una cadena de llamadas (una función que llama a otra) para ver esa estructura completa.

In [1]:
def obtener_factor_riesgo(tipo_poliza: str) -> float:
    factores = {"auto": 1.15, "hogar": 1.05, "vida": 1.30}
    return factores[tipo_poliza]  # KeyError si "tipo_poliza" no está en el diccionario


def calcular_prima_neta(poliza: dict) -> float:
    factor = obtener_factor_riesgo(poliza["tipo"])
    return round(poliza["prima_anual"] * factor, 2)


poliza_con_typo = {"id": "P006", "tipo": "vidda", "prima_anual": 300.0}  # "vidda" en vez de "vida"

import traceback

try:
    calcular_prima_neta(poliza_con_typo)
except KeyError:
    traceback.print_exc()


Traceback (most recent call last):
  File "C:\Users\Arnau Ivern\AppData\Local\Temp\ipykernel_42864\3430478007.py", line 16, in <module>
    calcular_prima_neta(poliza_con_typo)
    ~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^
  File "C:\Users\Arnau Ivern\AppData\Local\Temp\ipykernel_42864\3430478007.py", line 7, in calcular_prima_neta
    factor = obtener_factor_riesgo(poliza["tipo"])
  File "C:\Users\Arnau Ivern\AppData\Local\Temp\ipykernel_42864\3430478007.py", line 3, in obtener_factor_riesgo
    return factores[tipo_poliza]  # KeyError si "tipo_poliza" no está en el diccionario
           ~~~~~~~~^^^^^^^^^^^^^
KeyError: 'vidda'


Leyendo este traceback de arriba abajo: la primera parada es la línea donde se llamó a `calcular_prima_neta(...)`; la segunda es la línea `factor = obtener_factor_riesgo(...)` **dentro** de esa función; la tercera (la última, la más relevante) es `return factores[tipo_poliza]` dentro de `obtener_factor_riesgo`, que es donde realmente se produjo el fallo. La última línea, `KeyError: 'vidda'`, dice exactamente qué llave faltaba. Con esa única línea ya sabes dónde mirar: el diccionario `factores` no tiene la llave `"vidda"` — y, si sigues investigando, verás que el problema real no está en `obtener_factor_riesgo` (que está bien escrita) sino en el dato de entrada: alguien escribió `"vidda"` en vez de `"vida"`.

> ⚠️ **Error típico**: mirar solo la primera línea del traceback (la más externa) y darse por vencido ahí, en vez de bajar hasta la última parada — que es casi siempre donde está la causa real, no el síntoma.

## 2. Galería de errores frecuentes en condicionales y bucles

Ya viste hoy `IndentationError`, `SyntaxError` (Notebook 01) y `StopIteration` (Notebook 03). Aquí tienes el resto de la familia que te vas a encontrar constantemente a partir de ahora, todos provocados de forma controlada con `try/except` para poder mostrarlos sin romper la ejecución del notebook.

In [2]:
# NameError: usar una variable que no existe (todavía) en el entorno actual.
# Muy típico al reescribir una condición y cambiar el nombre de una variable
# sin actualizar el resto del bloque.
try:
    if prima_sin_definir > 100:
        print("Prima alta")
except NameError as e:
    print(f"{type(e).__name__}: {e}")


NameError: name 'prima_sin_definir' is not defined


In [3]:
# TypeError: mezclar tipos que Python no sabe combinar de esa forma.
# Muy típico al construir un mensaje concatenando texto y un número directamente.
try:
    prima = 480.50
    mensaje = "La prima es: " + prima  # falta convertir prima a texto, o usar un f-string
except TypeError as e:
    print(f"{type(e).__name__}: {e}")


TypeError: can only concatenate str (not "float") to str


In [4]:
# IndexError: pedir una posición de una lista que no existe.
# Muy típico en un bucle "for i in range(...)" con un límite mal calculado.
cartera = ["P001", "P002", "P003"]
try:
    print(cartera[5])
except IndexError as e:
    print(f"{type(e).__name__}: {e}")


IndexError: list index out of range


In [5]:
# ZeroDivisionError: dividir entre algo que, para algún caso concreto de tus datos, vale cero.
# Muy típico al calcular una media o una tasa sin comprobar antes si el denominador es válido.
siniestros_pagados = 0
polizas_activas = 0  # cartera vacía en este segmento, por ejemplo
try:
    frecuencia_siniestralidad = siniestros_pagados / polizas_activas
except ZeroDivisionError as e:
    print(f"{type(e).__name__}: {e}")


ZeroDivisionError: division by zero


Fíjate en el patrón común: **el mensaje de error casi siempre dice literalmente qué esperaba Python y qué ha encontrado en su lugar** (`'prima_sin_definir' is not defined`, `unsupported operand type(s)`, `list index out of range`, `division by zero`). Leer ese mensaje con calma, en vez de reaccionar a "hay un error, algo está mal", resuelve la mayoría de casos sin necesidad de ayuda externa.

## 3. Bloque de IA: debugging asistido — pegar el traceback completo, no un resumen

Cuando el mensaje de error no basta para entender el problema por ti mismo, Claude Code es una herramienta muy potente para interpretarlo — **si le das el traceback completo**, no un resumen tuyo tipo "me da un error con los diccionarios". El resumen ya es una interpretación (a veces equivocada) de lo que ha pasado; el traceback completo es el hecho en bruto.

Recupera el error de `calcular_prima_neta` de la sección 1 (el de `"vidda"`). Este es el prompt que le pasarías a Claude Code, con la anatomía de contexto + objetivo + restricciones + formato de salida que ya viste el Día 1:

In [6]:
prompt_debugging = """
Contexto: en dia_03_control_de_flujo/05_teoria_errores_y_debugging.ipynb tengo dos
funciones, obtener_factor_riesgo(tipo_poliza) y calcular_prima_neta(poliza), y al
llamar a calcular_prima_neta con poliza_con_typo obtengo este traceback completo:

Traceback (most recent call last):
  File "...", line 12, in <module>
    calcular_prima_neta(poliza_con_typo)
  File "...", line 7, in calcular_prima_neta
    factor = obtener_factor_riesgo(poliza["tipo"])
  File "...", line 3, in obtener_factor_riesgo
    return factores[tipo_poliza]
KeyError: 'vidda'

Objetivo: entender por que aparece el error y como corregirlo sin cambiar el
comportamiento de obtener_factor_riesgo para el resto de tipos de poliza validos.

Restricciones: no me des directamente el codigo corregido. Antes, explicame que
significa KeyError en este caso concreto y ayudame a decidir si el fallo esta en
el diccionario "factores" o en el dato de entrada "poliza_con_typo".

Formato de salida: una explicacion breve del error linea a linea del traceback,
seguida de una pregunta que deberia responder yo mismo antes de tocar el codigo.
"""
print(prompt_debugging)



Contexto: en dia_03_control_de_flujo/05_teoria_errores_y_debugging.ipynb tengo dos
funciones, obtener_factor_riesgo(tipo_poliza) y calcular_prima_neta(poliza), y al
llamar a calcular_prima_neta con poliza_con_typo obtengo este traceback completo:

Traceback (most recent call last):
  File "...", line 12, in <module>
    calcular_prima_neta(poliza_con_typo)
  File "...", line 7, in calcular_prima_neta
    factor = obtener_factor_riesgo(poliza["tipo"])
  File "...", line 3, in obtener_factor_riesgo
    return factores[tipo_poliza]
KeyError: 'vidda'

Objetivo: entender por que aparece el error y como corregirlo sin cambiar el
comportamiento de obtener_factor_riesgo para el resto de tipos de poliza validos.

Restricciones: no me des directamente el codigo corregido. Antes, explicame que
significa KeyError en este caso concreto y ayudame a decidir si el fallo esta en
el diccionario "factores" o en el dato de entrada "poliza_con_typo".

Formato de salida: una explicacion breve del error lin

**Qué se espera obtener**: una explicación que recorra el traceback de abajo arriba (o de arriba abajo, según prefiera explicarlo) señalando la línea exacta donde salta el `KeyError`, una aclaración de que `factores[tipo_poliza]` falla porque `"vidda"` no es una llave del diccionario, y una pregunta guía del estilo "¿el diccionario `factores` debería tener más llaves, o el dato `poliza_con_typo` tiene un error tipográfico?" — no directamente la línea de código corregida.

**Qué debes verificar antes de darlo por bueno** (esta es la parte que realmente se evalúa, según `docs/uso_ia_en_el_curso.md`):

- Que la explicación señale la línea **correcta** del traceback como origen real del fallo (la última parada, `return factores[tipo_poliza]`), no una línea distinta por casualidad.
- Que la corrección que finalmente apliques tú mismo — no la IA — sea coherente con la causa real: en este caso, el diccionario `factores` está bien y el dato de entrada tiene un error tipográfico (`"vidda"` en vez de `"vida"`); "arreglarlo" añadiendo `"vidda"` como una llave más del diccionario sería tapar el síntoma, no corregir la causa.
- Que, tras aplicar la corrección, **ejecutes de nuevo el código** y compruebes que el resultado es el esperado (`calcular_prima_neta(poliza_con_typo)` debería devolver `390.0` una vez corregido el tipo a `"vida"`) — nunca des un error por resuelto solo porque la explicación "suena razonable".

In [7]:
# Comprobación de control: aplicamos la corrección real (el dato tenía el typo, no el diccionario)
# y verificamos que el resultado es el esperado antes de dar el caso por cerrado.
poliza_corregida = {"id": "P006", "tipo": "vida", "prima_anual": 300.0}
resultado = calcular_prima_neta(poliza_corregida)
print("Prima neta corregida:", resultado)
assert resultado == 390.0, "El resultado no coincide con lo esperado -- revisar antes de dar por bueno"


Prima neta corregida: 390.0


**Para ti:** provoca tú mismo (con `try/except`, como en toda esta sección) un `TypeError` distinto al del ejemplo — por ejemplo, sumando una lista y un entero — y escribe, en una celda de markdown, el prompt que le pasarías a Claude Code siguiendo las cuatro partes (contexto, objetivo, restricciones, formato de salida) para que te ayude a entenderlo sin darte la solución directa.

**Para ti:** el siguiente fragmento tiene un error de lógica (no de sintaxis: se ejecuta sin lanzar ningún traceback) que hace que la lista `polizas_caras` salga vacía cuando no debería. Encuéntralo tú antes de ejecutar nada, corrígelo, y solo entonces ejecuta la celda para comprobar que ahora sí aparecen las tres pólizas con prima superior a 300 €.

```python
primas = [480.50, 210.00, 650.00, 395.75, 180.00]
polizas_caras = [p for p in primas if p > 1000]  # pista: compara el umbral con los datos reales
print(polizas_caras)
```

In [8]:
primas = [480.50, 210.00, 650.00, 395.75, 180.00]
polizas_caras = [p for p in primas if p > 300]  # corregido: el umbral original (1000) era mayor que cualquier prima de la muestra
print(polizas_caras)
assert len(polizas_caras) == 3


[480.5, 650.0, 395.75]


## Resumen

- Un traceback se lee de arriba abajo: cada parada es una llamada; la última parada (la más abajo) es casi siempre donde está la causa real del fallo, no solo el síntoma.
- La última línea (`TipoDeError: mensaje`) dice el qué; el mensaje suele ser literal sobre qué esperaba Python y qué encontró — vale la pena leerlo entero antes de buscar ayuda externa.
- `NameError`, `TypeError`, `IndexError`, `ZeroDivisionError` y `KeyError` son los errores en tiempo de ejecución más comunes al trabajar con condicionales y bucles; se suman a `IndentationError`, `SyntaxError` (errores de sintaxis, detectados antes de ejecutar) y `StopIteration` vistos en notebooks anteriores.
- No todos los fallos lanzan una excepción: un error de lógica (como una comprehension con un umbral mal puesto, o un `range()` mal calculado) puede ejecutarse sin ningún traceback y dar igualmente un resultado incorrecto — hay que validar el resultado, no solo la ausencia de errores.
- Para pedir ayuda a Claude Code con un error, pega el **traceback completo**, no un resumen tuyo: el resumen ya es una interpretación que puede estar sesgada o incompleta.
- Un buen prompt de debugging sigue teniendo las cuatro partes del Día 1 (contexto, objetivo, restricciones, formato de salida) — aquí la restricción clave suele ser "no me des la solución directa, ayúdame a entender la causa".
- Antes de dar un error por resuelto: comprueba que la explicación señala la causa correcta (no solo una plausible), aplica tú mismo la corrección, y **vuelve a ejecutar** para verificar el resultado con un caso de control conocido — nunca te fíes de una explicación solo porque "suena razonable" (`docs/uso_ia_en_el_curso.md`).

**Siguiente:** `06_ejercicios.ipynb` — practica todo lo visto hoy antes de ver las soluciones en `07_soluciones.ipynb`.